# 04 — Pandas, EDA y pipelines

## Motivación

En las sesiones anteriores los datos llegaban ya como matrices numéricas: cada loader
de scikit-learn entrega un array donde cada columna es un número. Los datos reales
llegan en tablas, con columnas de texto, tipos mixtos y valores faltantes.
**pandas** es la librería para manipular esas tablas antes de que lleguen a un modelo.

Esta sesión tiene dos partes. Primero, la mecánica de pandas sobre California Housing,
un dataset ya conocido, para que el esfuerzo se concentre en la herramienta y no en
entender datos nuevos. Después, un dataset con **características categóricas reales**
(tráfico de red, con protocolos y servicios como texto) para construir un pipeline de
preprocesamiento completo, y cerrar con el concepto que ese pipeline existe para
evitar: la fuga de información entre train y test.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_DIR = Path("../../datos")

rng = np.random.default_rng(seed=42)

## 1. Cargar e inspeccionar

`fetch_california_housing(as_frame=True)` ya entrega un `DataFrame`. Lo hemos usado
así desde la sesión 02 sin detenernos en qué es. Un `DataFrame` es una tabla: filas
indexadas, columnas con nombre y con su propio tipo de dato. Antes de modelar, tres
preguntas de inspección: ¿qué forma tiene la tabla?, ¿qué tipo es cada columna?, ¿qué
rango de valores trae cada una?

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(data_home=DATA_DIR, as_frame=True)
df = housing.frame

print(f"shape: {df.shape}")
df.info()

In [ ]:
df.describe()

`describe()` ya adelanta algo que la sesión 02 mostró como gráfica: `AveRooms` tiene un
máximo de más de 100 habitaciones promedio por vivienda, y `AveOccup` un máximo de
más de 1,000 personas por vivienda, valores atípicos reales que conviene tener
presentes al interpretar cualquier resultado.

## 2. Selección y filtrado

Dos formas de indexar un `DataFrame`:

- `df.loc[filas, columnas]`: por **etiqueta** (nombre de columna, índice de fila).
- `df.iloc[filas, columnas]`: por **posición** entera, igual que numpy.

El filtrado booleano es el mismo concepto que las máscaras de numpy de la sesión 01,
aplicado a columnas con nombre: `df[condición]` selecciona las filas donde la
condición es verdadera.

In [ ]:
print(df.loc[0:2, ["MedInc", "MedHouseVal"]])
print()
print(df.iloc[0:3, 0:2])
print()

distritos_caros = df[df["MedHouseVal"] > 4.5]
print(f"distritos con valor > $450k: {len(distritos_caros)} de {len(df)}")

## 3. Agrupar y agregar

`groupby` sigue el patrón *dividir → aplicar → combinar*: divide la tabla en grupos
según una columna, aplica una función a cada grupo, combina los resultados en una
tabla nueva. `pd.cut` convierte una columna numérica en categorías por rango, útil
para crear el grupo cuando no existe una columna categórica de origen.

In [ ]:
df["antiguedad_categoria"] = pd.cut(
    df["HouseAge"], bins=[0, 15, 30, 52], labels=["nueva", "media", "antigua"]
)

df.groupby("antiguedad_categoria", observed=True)["MedHouseVal"].agg(["mean", "count"])

El valor mediano de vivienda crece con la antigüedad de la construcción en este
dataset, contrario a la intuición de que "nuevo" vale más. Una hipótesis posible:
viviendas antiguas concentradas en zonas urbanas consolidadas, más caras. La tabla no
permite confirmarla; distinguir correlación de explicación es parte del trabajo de
EDA.

## 4. Correlación

El coeficiente de correlación de Pearson entre cada par de columnas, calculado con
`df.corr()`, resume $n^2$ relaciones en una sola tabla (o, mejor, en un mapa de calor).

In [ ]:
corr = df.drop(columns="antiguedad_categoria").corr(numeric_only=True)

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.columns,
    colorscale="RdBu", zmid=0, zmin=-1, zmax=1,
    text=corr.round(2).values, texttemplate="%{text}", textfont=dict(size=9),
))
fig.update_layout(
    title="Correlación entre variables de California Housing",
    template="plotly_white",
    height=550,
)
fig.show()

`AveRooms` y `AveBedrms` correlacionan en 0.85, información casi redundante entre
ambas. `Latitude` y `Longitude` correlacionan en -0.92, consecuencia de la forma
alargada del estado de California en el mapa. Ninguna característica correlaciona por
encima de 0.7 con el objetivo `MedHouseVal`, consistente con lo que ya se vio en la
sesión 02: el ingreso solo explica una fracción del precio.

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("AveRooms vs AveBedrms (r=0.85)", "Longitude vs Latitude (r=-0.92)"),
)

fig.add_trace(
    go.Scatter(x=df["AveRooms"], y=df["AveBedrms"], mode="markers",
                marker=dict(size=4, opacity=0.35, color="#636EFA"), showlegend=False),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=df["Longitude"], y=df["Latitude"], mode="markers",
                marker=dict(size=4, opacity=0.35, color="#636EFA"), showlegend=False),
    row=1, col=2,
)
fig.update_xaxes(title_text="AveRooms", range=[0, 15], row=1, col=1)
fig.update_yaxes(title_text="AveBedrms", range=[0, 5], row=1, col=1)
fig.update_xaxes(title_text="Longitude", row=1, col=2)
fig.update_yaxes(title_text="Latitude", row=1, col=2)
fig.update_layout(
    title="Las dos correlaciones más fuertes del mapa de calor, vistas directamente",
    template="plotly_white",
    height=450,
)
fig.show()

El panel derecho reproduce el contorno de California, evidencia visual de que la
correlación de -0.92 entre longitud y latitud es un artefacto geométrico del mapa. El
panel izquierdo muestra una nube alineada a lo largo de la diagonal: el número de
habitaciones y el de recámaras promedio crecen juntos, lo que explica la correlación
de 0.85 entre ambas columnas. El eje de `AveRooms` se recorta en 15 para facilitar la
lectura; los distritos con valores superiores, identificados en la sección 1, quedan
fuera del gráfico.

## 5. Valores faltantes

Los datasets curados de scikit-learn no traen valores faltantes. Para practicar cómo
tratarlos, se simulan aquí sobre una copia de California Housing. Se declara
explícitamente porque en un dataset real esta celda no existiría.

In [ ]:
df_num = df.drop(columns="antiguedad_categoria").copy()

mask = rng.random(df_num.shape) < 0.05  # ~5% de celdas, simulado
df_missing = df_num.mask(mask)

df_missing.isna().sum()

`SimpleImputer` reemplaza cada valor faltante por una estadística de su columna
(aquí, la mediana). Como cualquier transformador de scikit-learn, sigue el patrón
`fit`/`transform`: `fit` calcula la mediana de cada columna, `transform` la usa para
rellenar. Esto importa para lo que sigue: `fit` debe correr **solo sobre train**, para
no filtrar información del test a la imputación, la primera aparición del problema
que cierra la sesión.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
df_imputed = pd.DataFrame(
    imputer.fit_transform(df_missing), columns=df_missing.columns
)

print(f"faltantes antes:  {df_missing.isna().sum().sum()}")
print(f"faltantes después: {df_imputed.isna().sum().sum()}")

## 6. Características categóricas: KDD Cup 99

Hasta ahora, cada dataset del curso ha sido puramente numérico. Los datos reales traen
**categorías**: texto que identifica un tipo, no una cantidad. **KDD Cup 99** es un
dataset de tráfico de red para detección de intrusiones: cada fila es una conexión,
con columnas numéricas (bytes transmitidos, duración) y columnas categóricas
(`protocol_type`, `service`, `flag`): el protocolo de red, el servicio y el estado de
la conexión.

In [ ]:
from sklearn.datasets import fetch_kddcup99

kdd = fetch_kddcup99(subset="SA", percent10=True, as_frame=True, data_home=DATA_DIR)
net = kdd.frame

print(f"shape: {net.shape}")
net.dtypes.value_counts()

Todas las columnas llegan con dtype `object`, incluidas las que son numéricas: así
entrega los datos este loader. Dos limpiezas son necesarias antes de usar la tabla:

1. Las columnas categóricas contienen literales `bytes` (`b'tcp'`, no `'tcp'`), se
   decodifican a texto con `.str.decode("utf-8")`.
2. Las columnas numéricas quedaron con dtype genérico, se convierten con
   `.astype(float)`.

In [ ]:
categorical_cols = ["protocol_type", "service", "flag"]
numeric_cols = [c for c in net.columns if c not in categorical_cols + ["labels"]]

for col in categorical_cols + ["labels"]:
    net[col] = net[col].str.decode("utf-8")
net[numeric_cols] = net[numeric_cols].astype(float)

net[categorical_cols + ["labels"]].head(3)

Con los tipos corregidos, `value_counts()` describe cada categórica, y la distribución
de `labels` muestra el desbalance del dataset: la enorme mayoría de las conexiones son
normales.

In [ ]:
for col in categorical_cols:
    print(f"{col}: {net[col].nunique()} valores — {net[col].unique()[:5].tolist()}")

print()
proporcion_ataques = (net["labels"] != "normal.").mean()
print(f"proporción de conexiones que son ataques: {proporcion_ataques:.4f}")

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("protocol_type", "flag", "service (15 más frecuentes)", "labels (escala log)"),
)

protocol_counts = net["protocol_type"].value_counts()
fig.add_trace(
    go.Bar(x=protocol_counts.index, y=protocol_counts.values,
           marker_color="#636EFA", showlegend=False),
    row=1, col=1,
)

flag_counts = net["flag"].value_counts()
fig.add_trace(
    go.Bar(x=flag_counts.index, y=flag_counts.values,
           marker_color="#636EFA", showlegend=False),
    row=1, col=2,
)

service_counts = net["service"].value_counts().head(15)
fig.add_trace(
    go.Bar(x=service_counts.index, y=service_counts.values,
           marker_color="#636EFA", showlegend=False),
    row=2, col=1,
)

label_counts = net["labels"].value_counts()
fig.add_trace(
    go.Bar(x=label_counts.index, y=label_counts.values,
           marker_color="#EF553B", showlegend=False),
    row=2, col=2,
)
fig.update_yaxes(type="log", row=2, col=2)
fig.update_xaxes(tickangle=45, row=2, col=1)
fig.update_xaxes(tickangle=45, row=2, col=2)

fig.update_layout(
    title="Distribución de las columnas categóricas y del objetivo (KDD Cup 99)",
    template="plotly_white",
    height=750,
)
fig.show()

El panel de `labels` de arriba, en escala logarítmica porque en escala lineal las 13
categorías salvo `normal.` serían invisibles, hace evidente el desbalance: menos del
4% de las conexiones son ataques, repartidos además en 10 tipos distintos con
frecuencias muy dispares. Ese desbalance vuelve inútil una métrica como la proporción
de aciertos para evaluar el modelo, el tema central de la próxima sesión. Aquí se
define el objetivo de la forma más simple posible, binaria: ¿la conexión es un ataque
o no?

Los otros tres paneles muestran la asimetría propia de `service`: `http` domina con
más de 60,000 conexiones mientras la mayoría de los otros 50 servicios apenas suman
unas pocas: la razón por la que el panel se recorta a los 15 más frecuentes.

In [ ]:
X_net = net[categorical_cols + numeric_cols]
y_net = (net["labels"] != "normal.").astype(int)

## 7. Fuga de información (*data leakage*)

`StandardScaler`, `OneHotEncoder`, `SimpleImputer`: todos aprenden algo de los datos
durante `fit` (una media, las categorías que existen, una mediana) y lo aplican
durante `transform`. Si ese `fit` corre sobre la tabla completa, **antes** de separar
train y test, el conjunto de test influyó en una transformación que después se usa
para evaluar sobre ese mismo test. Información que debía permanecer fuera del
entrenamiento terminó incorporada indirectamente al modelo. Esto se llama **fuga de
información**, y el resultado es una estimación de desempeño optimista que no se
sostiene en producción.

El orden correcto es: dividir train/test primero, ajustar cada transformador
**solo sobre train**, aplicar la misma transformación (ya ajustada) sobre test. Un
`Pipeline` automatiza exactamente ese orden: encadena transformadores y modelo en un
único objeto donde `fit` recibe solo los datos que se le pasan explícitamente, y
`cross_val_score`/`GridSearchCV` repiten ese `fit` en cada fold, evitando la fuga
incluso dentro de la validación cruzada.

## 8. `ColumnTransformer`: distintas columnas, distinto preprocesamiento

Un modelo lineal opera sobre números: `protocol_type` debe convertirse a una
representación numérica antes de entrar a `LogisticRegression`. La conversión más
directa, un entero por categoría (`tcp`→0, `udp`→1, `icmp`→2), hace que el modelo
interprete esos números con su orden y distancia habituales: implica que `icmp` está
el doble de lejos de `tcp` que `udp`, una relación que no existe entre protocolos de
red, donde ninguna categoría es "mayor" que otra. **`OneHotEncoder`** evita esa
distancia inventada: crea una columna binaria por categoría, todas a la misma
distancia entre sí.

Las columnas numéricas no tienen ese problema (ya son números con una escala propia),
así que solo necesitan `StandardScaler`. `ColumnTransformer` aplica cada transformador
a su propio subconjunto de columnas y concatena el resultado. Se encadena con el
modelo dentro de un `Pipeline`, igual que `PolynomialFeatures` y `StandardScaler` se
encadenaron en la sesión 03.

In [ ]:
sample = pd.concat([
    net[net["protocol_type"] == p].head(3) for p in ["tcp", "udp", "icmp"]
])[["protocol_type"]].reset_index(drop=True)

sample["codigo_ordinal"] = pd.factorize(sample["protocol_type"])[0]
onehot_sample = pd.get_dummies(sample["protocol_type"])[["tcp", "udp", "icmp"]].astype(int)

filas = [f"{p} (#{i})" for i, p in enumerate(sample["protocol_type"])]

fig = make_subplots(rows=1, cols=2, subplot_titles=("Codificación ordinal", "One-hot encoding"))
fig.add_trace(
    go.Heatmap(z=sample[["codigo_ordinal"]].values, x=["código"], y=filas,
               colorscale="Viridis", showscale=False,
               text=sample[["codigo_ordinal"]].values, texttemplate="%{text}"),
    row=1, col=1,
)
fig.add_trace(
    go.Heatmap(z=onehot_sample.values, x=list(onehot_sample.columns), y=filas,
               colorscale=[[0, "#F0F0F0"], [1, "#636EFA"]], showscale=False,
               text=onehot_sample.values, texttemplate="%{text}"),
    row=1, col=2,
)
fig.update_layout(
    title="Un entero implica un orden entre categorías que no existe; one-hot no",
    template="plotly_white",
    height=400,
)
fig.show()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X_net, y_net, test_size=0.2, random_state=42, stratify=y_net
)

preprocesamiento = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("numericas", StandardScaler(), numeric_cols),
])

pipeline = Pipeline([
    ("preprocesamiento", preprocesamiento),
    ("clasificador", LogisticRegression(max_iter=1000)),
])

scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring="accuracy")
print(f"accuracy por fold: {np.round(scores, 4)}")
print(f"accuracy promedio (CV): {scores.mean():.4f}")

pipeline.fit(X_train, y_train)
print(f"accuracy en test:       {pipeline.score(X_test, y_test):.4f}")

El pipeline completo (decodificación ya hecha antes, codificación de categóricas,
escalado de numéricas, modelo) corre de principio a fin sin que el test participe en
ningún ajuste. La accuracy sale cercana a 1.0: con menos del 4% de ejemplos positivos,
un modelo que acertara siempre "no es ataque" ya obtendría más de 96%. La accuracy
no distingue eso de un modelo que efectivamente detecta ataques. Las métricas
adecuadas para esta situación son el tema de la sesión 05.

## Ejercicio

Trabaja en una copia de este notebook dentro de `mi-trabajo/`.

1. **Agregación.** Sobre California Housing, agrupa los distritos en 4 categorías de
   ingreso (`pd.cut` sobre `MedInc`) y calcula, para cada categoría, el valor mediano
   de vivienda promedio y el número de distritos. Grafica el resultado con un `go.Bar`.

2. **Imputación y su efecto.** Repite la simulación de faltantes con tres tasas
   distintas (5%, 20%, 40%) y compara, para cada una, el RMSE de una regresión lineal
   sobre los datos imputados contra el RMSE sobre los datos originales sin faltantes.
   ¿En qué punto la imputación deja de ser suficiente?

3. **Reto — otro pipeline.** Sobre KDD Cup 99, cambia `LogisticRegression` por
   `RandomForestClassifier` dentro del mismo `Pipeline`, sin cambiar nada más.
   Verifica que el `ColumnTransformer` es reutilizable entre modelos distintos.